# ServLoci SDK Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivikasavnish/algo-trading-notebooks/blob/main/notebooks/01_servloci_sdk_quickstart.ipynb)

The four ways to use the `ServLoci` Python class.

Part 01 of 35 in the [ServLoci algo/options trading notebook series](https://comm.servloci.in/docs) — full index in `notebooks/README.md`.

## Setup

In [ ]:
# Get your dedicated static IPv6 + SOCKS5 credentials free:
#   https://comm.servloci.in/register        (or /auth/google?free=1 for an instant trial)
# Your api_key / api_secret pair shows up in the portal after signup:
#   https://comm.servloci.in/user
!pip install -q "requests[socks]"
!curl -sL https://comm.servloci.in/sdk/servloci.py -o servloci.py

import os
from servloci import ServLoci

SERVLOCI_API_KEY = os.environ.get("SERVLOCI_API_KEY", "dhan:1000000001")   # broker:client_id
SERVLOCI_API_SECRET = os.environ.get("SERVLOCI_API_SECRET", "")            # from the portal — leave blank to run this notebook in demo mode

sl = None
if SERVLOCI_API_SECRET:
    sl = ServLoci(api_key=SERVLOCI_API_KEY, api_secret=SERVLOCI_API_SECRET)
    print("ServLoci configured:", sl.host, sl.port)
else:
    print("SERVLOCI_API_SECRET not set — running in demo mode (no live proxy calls).")

## Two ways to route traffic, and why the difference matters

A `requests.Session` (and the connection pool underneath it) is created once
and reused for every call made through it — brokers' Python SDKs each build
one internally the moment you import them. That means *when* you enable a
proxy relative to that import is not cosmetic; it decides whether the
broker's own HTTP calls ever see it.

The SDK gives you two strategies for that reason:

- **Scoped**: `session()` returns a fresh `requests.Session` with the proxy
  already set. Use it for your own `requests` calls; it never touches code you
  don't own.
- **Process-wide**: `attach()` monkey-patches `requests.Session.__init__` so
  that *every* session created afterwards — including the one a broker SDK
  builds internally on import — picks up the proxy automatically. This is the
  only way to route a third-party SDK's traffic without editing its source.

`proxy_url()` / `proxies()` and `export_env()` exist for tools that don't use
`requests` at all — a raw `curl`, an `httpx` client, or any process that reads
standard `HTTP_PROXY`/`HTTPS_PROXY` environment variables.

In [ ]:
from servloci import ServLoci, configure

demo = ServLoci(api_key="dhan:1000000001", api_secret="demo-secret")
print("proxy_url():", demo.proxy_url())
print("proxies():  ", demo.proxies())

try:
    ServLoci(api_key="", api_secret="")
except ValueError as e:
    print("Missing credentials raise ValueError:", e)

if sl:
    sl.export_env()
    print("HTTPS_PROXY set in os.environ:", "HTTPS_PROXY" in os.environ)
    s = sl.session()
    print("session() proxies:", s.proxies)

`attach()` must run **before** you import a broker SDK — dhanhq, kiteconnect,
growwapi and fyers-apiv3 all create a `requests.Session` at import time, so
patching afterwards is a no-op for their traffic even though it still works
for your own. The one-liner `servloci.configure(api_key, api_secret)` does
`ServLoci(...).attach()` in a single call, and is the pattern used in
notebooks 02-05 below.

---

« Previous: [Get Your Static IP & Verify It](00_get_static_ip_and_verify.ipynb)  
Next: [Broker Auth: Zerodha (Kite Connect)](02_broker_auth_zerodha_kite.ipynb) »

Try the concepts above interactively: [Options Strategy Builder](https://comm.servloci.in/tools/strategy-builder) · [Docs](https://comm.servloci.in/docs) · [Get your static IP](https://comm.servloci.in/register)